# NME Portfolio - Earned Value Deep Learning Model

## Objective
Predict **Earned Value (EV)** for pharmaceutical NME projects using a TensorFlow neural network.

## Features (Independent Variables)
- **Planned Value (PV)** — budgeted cost of work scheduled
- **Actual Cost (AC)** — what's actually been spent
- **Budget at Completion (BAC)**
- **% Complete** — physical or reported completion percentage
- **Schedule Performance Index (SPI)** — EV/PV from prior periods
- **Cost Performance Index (CPI)** — EV/AC from prior periods
- **Schedule Variance (SV)** — from prior periods
- **Cost Variance (CV)** — from prior periods

## Target Variable
- **Earned Value (EV)** — the value of work actually performed

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Data Preparation

Generate synthetic EVM data for NME pharmaceutical projects.
In production, this would connect to the PostgreSQL database using the `v_nme_evm` view.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
n_samples = 500

# Therapeutic areas from the NME portfolio schema
therapeutic_areas = [
    'ONCOLOGY', 'CARDIOVASCULAR', 'NEUROLOGY', 'IMMUNOLOGY',
    'INFECTIOUS_DISEASE', 'METABOLIC', 'RESPIRATORY', 'RARE_DISEASE'
]

# NME statuses
nme_statuses = [
    'PRECLINICAL', 'IND_FILED', 'PHASE_1', 'PHASE_2', 'PHASE_3', 'NDA_FILED'
]

# Generate synthetic EVM data
data = {
    # Project identifiers
    'project_id': [f'PRJ-{str(i).zfill(4)}' for i in range(n_samples)],
    'therapeutic_area': np.random.choice(therapeutic_areas, n_samples),
    'nme_status': np.random.choice(nme_statuses, n_samples),
    
    # Budget at Completion (BAC) - pharmaceutical projects range $1M to $500M
    'bac': np.random.uniform(1_000_000, 500_000_000, n_samples),
    
    # Percent Complete (0-100%)
    'percent_complete': np.random.uniform(0, 100, n_samples),
    
    # Time-based factors
    'planned_duration_months': np.random.randint(6, 60, n_samples),
    'elapsed_months': np.random.randint(1, 48, n_samples),
}

df = pd.DataFrame(data)

# Ensure elapsed_months <= planned_duration_months
df['elapsed_months'] = df.apply(
    lambda x: min(x['elapsed_months'], x['planned_duration_months']), axis=1
)

# Calculate time percentage
df['time_percent'] = (df['elapsed_months'] / df['planned_duration_months']) * 100

# Planned Value (PV) = BAC * (elapsed_time / total_duration)
df['pv'] = df['bac'] * (df['time_percent'] / 100)

# Add realistic noise and calculate EV based on performance
# EV = BAC * percent_complete / 100 (with some variation)
performance_factor = np.random.uniform(0.7, 1.3, n_samples)  # Some projects over/under perform
df['ev'] = df['bac'] * (df['percent_complete'] / 100) * np.clip(performance_factor, 0.5, 1.5)

# Actual Cost (AC) - varies based on efficiency
cost_efficiency = np.random.uniform(0.8, 1.4, n_samples)  # Cost overruns/underruns
df['ac'] = df['ev'] * cost_efficiency

# Calculate performance indices (prior period values - with some lag)
# SPI = EV / PV (Schedule Performance Index)
df['spi_prior'] = np.where(df['pv'] > 0, df['ev'] / df['pv'], 1.0)
df['spi_prior'] = df['spi_prior'] * np.random.uniform(0.9, 1.1, n_samples)  # Add historical variation
df['spi_prior'] = np.clip(df['spi_prior'], 0.3, 2.0)  # Realistic bounds

# CPI = EV / AC (Cost Performance Index)
df['cpi_prior'] = np.where(df['ac'] > 0, df['ev'] / df['ac'], 1.0)
df['cpi_prior'] = df['cpi_prior'] * np.random.uniform(0.9, 1.1, n_samples)  # Add historical variation
df['cpi_prior'] = np.clip(df['cpi_prior'], 0.3, 2.0)  # Realistic bounds

# Schedule Variance (SV) = EV - PV (from prior period)
df['sv_prior'] = (df['ev'] - df['pv']) * np.random.uniform(0.85, 1.15, n_samples)

# Cost Variance (CV) = EV - AC (from prior period)
df['cv_prior'] = (df['ev'] - df['ac']) * np.random.uniform(0.85, 1.15, n_samples)

# Display data summary
print(f"Dataset shape: {df.shape}")
print(f"\nSample data:")
display(df.head(10))

In [ ]:
# Statistical summary of key EVM features
evm_columns = ['bac', 'pv', 'ev', 'ac', 'percent_complete', 'spi_prior', 'cpi_prior', 'sv_prior', 'cv_prior']
print("EVM Feature Statistics:")
display(df[evm_columns].describe())

In [ ]:
# Visualize key relationships
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# EV vs PV
axes[0, 0].scatter(df['pv'], df['ev'], alpha=0.5, c='blue')
axes[0, 0].plot([0, df['pv'].max()], [0, df['pv'].max()], 'r--', label='EV=PV')
axes[0, 0].set_xlabel('Planned Value (PV)')
axes[0, 0].set_ylabel('Earned Value (EV)')
axes[0, 0].set_title('EV vs PV Relationship')
axes[0, 0].legend()

# EV vs AC
axes[0, 1].scatter(df['ac'], df['ev'], alpha=0.5, c='green')
axes[0, 1].plot([0, df['ac'].max()], [0, df['ac'].max()], 'r--', label='EV=AC')
axes[0, 1].set_xlabel('Actual Cost (AC)')
axes[0, 1].set_ylabel('Earned Value (EV)')
axes[0, 1].set_title('EV vs AC Relationship')
axes[0, 1].legend()

# EV vs BAC
axes[0, 2].scatter(df['bac'], df['ev'], alpha=0.5, c='purple')
axes[0, 2].set_xlabel('Budget at Completion (BAC)')
axes[0, 2].set_ylabel('Earned Value (EV)')
axes[0, 2].set_title('EV vs BAC Relationship')

# SPI Distribution
axes[1, 0].hist(df['spi_prior'], bins=30, color='orange', edgecolor='black')
axes[1, 0].axvline(x=1.0, color='red', linestyle='--', label='SPI=1.0')
axes[1, 0].set_xlabel('Schedule Performance Index (SPI)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('SPI Distribution')
axes[1, 0].legend()

# CPI Distribution
axes[1, 1].hist(df['cpi_prior'], bins=30, color='teal', edgecolor='black')
axes[1, 1].axvline(x=1.0, color='red', linestyle='--', label='CPI=1.0')
axes[1, 1].set_xlabel('Cost Performance Index (CPI)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('CPI Distribution')
axes[1, 1].legend()

# EV by Therapeutic Area
df.groupby('therapeutic_area')['ev'].mean().sort_values().plot(kind='barh', ax=axes[1, 2], color='coral')
axes[1, 2].set_xlabel('Average Earned Value (EV)')
axes[1, 2].set_title('Average EV by Therapeutic Area')

plt.tight_layout()
plt.show()

## 2. Feature Engineering and Preprocessing

In [ ]:
# Define features for the model
numerical_features = [
    'pv',              # Planned Value
    'ac',              # Actual Cost
    'bac',             # Budget at Completion
    'percent_complete', # % Complete
    'spi_prior',       # Schedule Performance Index (prior period)
    'cpi_prior',       # Cost Performance Index (prior period)
    'sv_prior',        # Schedule Variance (prior period)
    'cv_prior',        # Cost Variance (prior period)
]

categorical_features = [
    'therapeutic_area',
    'nme_status'
]

# Target variable
target = 'ev'

# Prepare feature matrix X and target vector y
X = df[numerical_features + categorical_features].copy()
y = df[target].values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nNumerical features: {numerical_features}")
print(f"Categorical features: {categorical_features}")

In [ ]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nProcessed feature shape: {X_train_processed.shape}")

# Scale target variable for better training stability
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

## 3. Build TensorFlow Neural Network Model

Architecture similar to Scientist_Selection_06 but adapted for regression:
- Input layer matching processed feature dimensions
- Dense hidden layers with ReLU activation
- Dropout for regularization
- Linear output for continuous EV prediction

In [ ]:
# Build the neural network model for EV regression
def build_ev_model(input_dim, learning_rate=0.001):
    """
    Build a neural network for Earned Value prediction.
    
    Architecture:
    - Input: EVM features (PV, AC, BAC, % Complete, SPI, CPI, SV, CV, categorical)
    - Hidden layers with ReLU activation and Dropout
    - Output: Single continuous value (Earned Value)
    """
    model = tf.keras.Sequential([
        # Input layer
        tf.keras.layers.Input(shape=(input_dim,)),
        
        # First hidden layer
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        
        # Second hidden layer
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        
        # Third hidden layer
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.1),
        
        # Output layer - single neuron for regression
        tf.keras.layers.Dense(1, activation='linear')
    ])
    
    # Compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',  # Mean Squared Error for regression
        metrics=['mae']  # Mean Absolute Error for interpretability
    )
    
    return model

# Build the model
input_dim = X_train_processed.shape[1]
model = build_ev_model(input_dim)

# Display model summary
model.summary()

## 4. Train the Model

In [ ]:
# Create TensorFlow datasets for efficient training
batch_size = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train_processed, y_train_scaled)
).shuffle(len(X_train_processed)).batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test_processed, y_test_scaled)
).batch(batch_size)

# Define callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train the model
print("Training the Earned Value prediction model...\n")
history = model.fit(
    train_dataset,
    epochs=100,
    validation_data=test_dataset,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
# Visualize training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Model Loss Over Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE', color='green')
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Absolute Error')
axes[1].set_title('Model MAE Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Model Evaluation

In [ ]:
# Make predictions on test set
y_pred_scaled = model.predict(X_test_processed).flatten()

# Inverse transform to get actual EV values
y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Calculate MAPE (Mean Absolute Percentage Error)
mape = np.mean(np.abs((y_test - y_pred) / np.where(y_test != 0, y_test, 1))) * 100

print("=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"Mean Absolute Error (MAE):     ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-squared (R2):                 {r2:.4f}")
print(f"Mean Absolute % Error (MAPE):  {mape:.2f}%")
print("=" * 50)

In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter plot: Predicted vs Actual
axes[0].scatter(y_test, y_pred, alpha=0.5, c='blue', edgecolors='k', linewidth=0.5)
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Earned Value ($)')
axes[0].set_ylabel('Predicted Earned Value ($)')
axes[0].set_title(f'Predicted vs Actual EV (R2 = {r2:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals histogram
residuals = y_test - y_pred
axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')
axes[1].grid(True, alpha=0.3)

# Residuals vs Predicted
axes[2].scatter(y_pred, residuals, alpha=0.5, c='green', edgecolors='k', linewidth=0.5)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Predicted Earned Value ($)')
axes[2].set_ylabel('Residual ($)')
axes[2].set_title('Residuals vs Predicted Values')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Permutation feature importance
from sklearn.inspection import permutation_importance

# Create a wrapper for the model to work with sklearn
class TFModelWrapper:
    def __init__(self, model, scaler):
        self.model = model
        self.scaler = scaler
    
    def predict(self, X):
        y_scaled = self.model.predict(X, verbose=0).flatten()
        return self.scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten()

wrapper = TFModelWrapper(model, target_scaler)

# Calculate permutation importance
print("Calculating feature importance (this may take a moment)...")
perm_importance = permutation_importance(
    wrapper, X_test_processed, y_test, 
    n_repeats=10, random_state=42, n_jobs=-1
)

# Get feature names from preprocessor
cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_features)

# Create importance dataframe
importance_df = pd.DataFrame({
    'feature': all_feature_names,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=True)

# Plot feature importance
plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance_mean'], 
         xerr=importance_df['importance_std'], color='steelblue')
plt.xlabel('Permutation Importance')
plt.ylabel('Feature')
plt.title('Feature Importance for Earned Value Prediction')
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
display(importance_df.tail(10).sort_values('importance_mean', ascending=False))

## 7. Make Predictions for New Projects

In [ ]:
# Example: Predict EV for new pharmaceutical projects
new_projects = pd.DataFrame({
    'pv': [50_000_000, 10_000_000, 200_000_000],           # Planned Value
    'ac': [48_000_000, 12_000_000, 180_000_000],           # Actual Cost
    'bac': [100_000_000, 25_000_000, 400_000_000],         # Budget at Completion
    'percent_complete': [45, 35, 55],                      # % Complete
    'spi_prior': [1.05, 0.85, 1.10],                       # Prior SPI
    'cpi_prior': [0.95, 0.80, 1.02],                       # Prior CPI
    'sv_prior': [2_500_000, -1_500_000, 10_000_000],       # Prior Schedule Variance
    'cv_prior': [-2_000_000, -3_000_000, 5_000_000],       # Prior Cost Variance
    'therapeutic_area': ['ONCOLOGY', 'NEUROLOGY', 'IMMUNOLOGY'],
    'nme_status': ['PHASE_2', 'PHASE_1', 'PHASE_3']
})

print("New Project Data:")
display(new_projects)

# Preprocess and predict
new_projects_processed = preprocessor.transform(new_projects)
predictions_scaled = model.predict(new_projects_processed, verbose=0).flatten()
predictions = target_scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

print("\n" + "=" * 60)
print("EARNED VALUE PREDICTIONS")
print("=" * 60)

for i, (_, row) in enumerate(new_projects.iterrows()):
    predicted_ev = predictions[i]
    expected_ev = row['bac'] * row['percent_complete'] / 100
    
    # Calculate predicted performance metrics
    pred_spi = predicted_ev / row['pv'] if row['pv'] > 0 else 0
    pred_cpi = predicted_ev / row['ac'] if row['ac'] > 0 else 0
    pred_sv = predicted_ev - row['pv']
    pred_cv = predicted_ev - row['ac']
    
    print(f"\nProject {i+1}: {row['therapeutic_area']} ({row['nme_status']})")
    print(f"  BAC: ${row['bac']:,.0f}")
    print(f"  Predicted EV: ${predicted_ev:,.0f}")
    print(f"  Simple EV (BAC x %): ${expected_ev:,.0f}")
    print(f"  Predicted SPI: {pred_spi:.3f} {'(Ahead)' if pred_spi > 1 else '(Behind)' if pred_spi < 1 else '(On Track)'}")
    print(f"  Predicted CPI: {pred_cpi:.3f} {'(Under Budget)' if pred_cpi > 1 else '(Over Budget)' if pred_cpi < 1 else '(On Budget)'}")
    print(f"  Schedule Variance: ${pred_sv:,.0f}")
    print(f"  Cost Variance: ${pred_cv:,.0f}")

## 8. Save Model for Production Use

In [ ]:
import joblib
import os

# Create models directory if it doesn't exist
models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)

# Save the Keras model
model.save(f'{models_dir}/nme_ev_model.keras')
print(f"Model saved to {models_dir}/nme_ev_model.keras")

# Save the preprocessor and scalers
joblib.dump(preprocessor, f'{models_dir}/ev_preprocessor.joblib')
joblib.dump(target_scaler, f'{models_dir}/ev_target_scaler.joblib')
print(f"Preprocessor saved to {models_dir}/ev_preprocessor.joblib")
print(f"Target scaler saved to {models_dir}/ev_target_scaler.joblib")

# Save feature configuration
feature_config = {
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'target': target
}
joblib.dump(feature_config, f'{models_dir}/ev_feature_config.joblib')
print(f"Feature config saved to {models_dir}/ev_feature_config.joblib")

In [ ]:
# Demonstration: Loading saved model for inference
print("Loading saved model...")

loaded_model = tf.keras.models.load_model(f'{models_dir}/nme_ev_model.keras')
loaded_preprocessor = joblib.load(f'{models_dir}/ev_preprocessor.joblib')
loaded_target_scaler = joblib.load(f'{models_dir}/ev_target_scaler.joblib')

# Test prediction with loaded model
test_project = pd.DataFrame({
    'pv': [75_000_000],
    'ac': [70_000_000],
    'bac': [150_000_000],
    'percent_complete': [50],
    'spi_prior': [1.0],
    'cpi_prior': [1.05],
    'sv_prior': [0],
    'cv_prior': [2_500_000],
    'therapeutic_area': ['ONCOLOGY'],
    'nme_status': ['PHASE_2']
})

test_processed = loaded_preprocessor.transform(test_project)
test_pred_scaled = loaded_model.predict(test_processed, verbose=0).flatten()
test_pred = loaded_target_scaler.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

print(f"\nLoaded model prediction for test project:")
print(f"  Predicted EV: ${test_pred[0]:,.0f}")
print(f"  Expected simple EV: ${150_000_000 * 0.5:,.0f}")

## Model Summary and Key Insights

### Architecture
- **Input Layer**: 8 numerical features + one-hot encoded categorical features
- **Hidden Layers**: 128 → 64 → 32 neurons with ReLU activation
- **Regularization**: Batch Normalization + Dropout (0.2, 0.1)
- **Output**: Single linear neuron for continuous EV prediction

### Key Features for EV Prediction
1. **Planned Value (PV)** - Strong predictor as EV = PV when on schedule
2. **Budget at Completion (BAC)** - Total budget bounds the EV
3. **% Complete** - Direct mathematical relationship with EV
4. **Prior SPI/CPI** - Historical performance trends
5. **Schedule/Cost Variance** - Deviation patterns from prior periods

### Production Considerations
- Connect to PostgreSQL using the `v_nme_evm` view for real data
- Retrain periodically as new project data becomes available
- Monitor model drift using validation metrics
- Consider ensemble methods for improved robustness